In [5]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc/>AGATCGGA')\
                  .named('template_pool').stylize(style='lower', which='contents')

mutated_pool = template_pool.stylize(region='cre', style='goldenrod')\
                            .mutagenize(region='cre',
                                        mutation_rate=0.1, 
                                        style='yellow bold underline lower',
                                        mode='random', 
                                        num_states=5, 
                                        prefix='mutagenize').named('mutated_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)

L = len('GGAAAGCGGGCAGTGAGCACACAGGA') 
recomb_pool = template_pool.recombine(region='cre', 
                                      num_breakpoints=3,
                                      sources=['A'*L, 'C'*L, 'G'*L, 'T'*L],
                                      styles=['palegreen', 'springgreen', 'limegreen', 'forestgreen'],
                                      style_by='order',
                                      mode='random',
                                      num_states=5,
                                      prefix='recomb').named('recomb_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)


deletion_pool = template_pool.stylize(region='cre', style='salmon')\
                             .deletion_scan(region='cre', 
                                            deletion_length=6, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style='red bold',
                                            prefix='delscan').named('deletion_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)

sites_pool=pp.from_seqs(['AAAAAA','TTTTTT'], 
                        mode='sequential', 
                        iter_order=-1).named('sites_pool')

insertion_pool = template_pool.stylize(region='cre', style='blue')\
                              .insertion_scan(region='cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              prefix='insscan',
                                              prefix_position='pos', 
                                              prefix_insert='ins',
                                              style='cyan bold').named('insertion_pool')
                              
shuffle_pool = template_pool.stylize(region='cre', style='purple')\
                            .shuffle_scan(region='cre', 
                                          shuffle_length=6, 
                                          shuffles_per_position=2,
                                          positions=slice(None, None, 5), 
                                          mode='sequential', 
                                          style='magenta bold',
                                          prefix='shufscan',
                                          prefix_position='pos',
                                          prefix_shuffle='shuf')
                            

combo_pool = pp.stack([mutated_pool, recomb_pool, deletion_pool, insertion_pool, shuffle_pool])\
    .named('stack_pool')\
    .insert_kmers(region='bc', mode='random', length=5, prefix='bc', style='green bold')\
    .named('combo_pool')\
    .stylize(which='tags', style='gray')

combo_pool.print_library(show_name=True, seed=42, suppress_styles=True)

NameError: name 'SeqStyle' is not defined

In [2]:
pp.print_named_colors()

Basic ANSI colors:
  red
  green
  yellow
  blue
  magenta
  cyan

CSS named colors:
aliceblue             antiquewhite          aqua                  aquamarine            
azure                 beige                 bisque                black                 
blanchedalmond        blueviolet            brown                 burlywood             
cadetblue             chartreuse            chocolate             coral                 
cornflowerblue        cornsilk              crimson               darkblue              
darkcyan              darkgoldenrod         darkgray              darkgreen             
darkgrey              darkkhaki             darkmagenta           darkolivegreen        
darkorange            darkorchid            darkred               darksalmon            
darkseagreen          darkslateblue         darkslategray         darkslategrey         
darkturquoise         darkviolet            deeppink              deepskyblue           
dimgray               dim

In [3]:
combo_pool.print_dag()

pool[27] (pool, n=50)
└── op[27]:stylize [mode=fixed, n=1]
    └── combo_pool (pool, n=50)
        └── op[26]:get_kmers [mode=random, n=50]
            └── stack_pool (pool, n=50)
                └── op[25]:stack [mode=sequential, n=5]
                    ├── pool[4] (pool, n=10)
                    │   └── op[4]:repeat [mode=sequential, n=2]
                    │       └── mutated_pool (pool, n=5)
                    │           └── op[3]:mutagenize [mode=random, n=5]
                    │               └── pool[2] (pool, n=1)
                    │                   └── op[2]:stylize [mode=fixed, n=1]
                    │                       └── pool[1] (pool, n=1)
                    │                           └── op[1]:stylize [mode=fixed, n=1]
                    │                               └── template_pool (pool, n=1)
                    │                                   └── op[0]:from_seq [mode=fixed, n=1]
                    ├── pool[10] (pool, n=10)
                 

Pool(id=27, name='pool[27]', op='op[27]:stylize', num_states=50)

In [4]:
combo_pool.mutagenize?

Signature:
combo_pool.mutagenize(
    region: Union[str, collections.abc.Sequence[numbers.Integral], NoneType] = None,
    num_mutations: Optional[numbers.Integral] = None,
    mutation_rate: Optional[numbers.Real] = None,
    allowed_chars: Optional[str] = None,
    style: Optional[str] = None,
    prefix: Optional[str] = None,
    mode: Literal['random', 'sequential', 'fixed'] = 'random',
    num_states: Optional[int] = None,
    iter_order: Optional[numbers.Real] = None,
) -> 'poolparty.pool.Pool'
Docstring:
Create a Pool that applies mutations to a sequence.

Parameters
----------
region : Union[str, Sequence[Integral], None], default=None
    Region to mutagenize. Can be a marker name (str), explicit interval [start, stop],
    or None to mutagenize entire sequence. Positions are region-relative.
num_mutations : Optional[Integral], default=None
    Fixed number of mutations to apply (mutually exclusive with mutation_rate).
mutation_rate : Optional[Real], default=None
    Probabili